# 84. Stellar Flares — Implementation / 항성 플레어 — 구현

**Paper**: Kowalski, A. F., "Stellar Flares", *Living Reviews in Solar Physics* **21**, 1 (2024). DOI: 10.1007/s41116-024-00039-4

This notebook implements six core quantitative concepts from the review:
1. Stellar flare energy power-law FFD $dN/dE \propto E^{-\alpha}$ with $\alpha \approx 2$
2. White-light flare continuum: $T_{\rm BB} = 9000$ K blackbody + Balmer jump
3. Neupert effect: $L_{\rm SXR}(t) \propto \int L_{\rm HXR}(t')\,dt'$
4. M-dwarf vs G-dwarf flare rates (GJ 1243 vs Sun)
5. Habitable-zone UV fluence from flares (Proxima b)
6. Exoplanet atmospheric photolysis from a superflare

이 노트북은 리뷰의 핵심 정량적 개념 6가지를 구현한다: 플레어 FFD 멱법칙, 9000 K 흑체+Balmer jump 백색광 연속광, Neupert 효과, M dwarf vs G dwarf 플레어율, 거주가능 영역 UV fluence, 슈퍼플레어에 의한 대기 광분해.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Physical constants (cgs)
H_PLANCK = 6.62607015e-27   # erg s
C_LIGHT = 2.99792458e10     # cm / s
K_BOLTZ = 1.380649e-16      # erg / K
SIGMA_SB = 5.670374419e-5   # erg cm^-2 s^-1 K^-4
L_SUN = 3.828e33            # erg / s
AU_CM = 1.495978707e13      # cm
PARSEC_CM = 3.0857e18       # cm
RSUN_CM = 6.957e10          # cm

## Part 1: Flare Frequency Distribution (FFD) / 플레어 빈도분포

Kepler observations of the dM4 star GJ 1243 give a power-law FFD with $\alpha = 2.008 \pm 0.002$ (Silverberg et al. 2016). We simulate a flare catalog drawn from this distribution and verify the slope via maximum-likelihood estimation (Clauset et al. 2009):
$$
\hat\alpha_{\rm ML} = 1 + N\left[\sum_{i=1}^{N}\ln(E_i/E_0)\right]^{-1}
$$

Kepler 관측의 GJ 1243 (dM4) 플레어는 $\alpha = 2.008$의 FFD를 보인다. 이 분포에서 플레어를 샘플링하고 최대우도 추정으로 기울기를 검증한다.

In [ ]:
def sample_power_law_flares(n_flares: int, alpha: float, e_min: float, e_max: float,
                            rng: np.random.Generator) -> np.ndarray:
    """Draw flare energies from a truncated power-law PDF.

    The differential FFD is n(E) dE = C E^(-alpha) dE for E_min <= E <= E_max.
    Uses inverse-CDF sampling.

    Args:
        n_flares: Number of flares to draw.
        alpha: Power-law index (differential, positive).
        e_min: Lower energy cutoff in erg.
        e_max: Upper energy cutoff in erg.
        rng: NumPy random Generator.

    Returns:
        Array of flare energies in erg.
    """
    u = rng.uniform(0.0, 1.0, size=n_flares)
    beta = 1.0 - alpha
    energies = (u * (e_max ** beta - e_min ** beta) + e_min ** beta) ** (1.0 / beta)
    return energies


def ml_power_law_index(energies: np.ndarray, e_min: float) -> tuple[float, float]:
    """Estimate the power-law index via maximum likelihood (Clauset+2009).

    Args:
        energies: Array of flare energies in erg.
        e_min: Low-energy reference threshold in erg.

    Returns:
        Tuple (alpha_hat, sigma_alpha).
    """
    n = len(energies)
    alpha_hat = 1.0 + n / np.sum(np.log(energies / e_min))
    sigma_alpha = (alpha_hat - 1.0) / np.sqrt(n)
    return alpha_hat, sigma_alpha


rng = np.random.default_rng(42)
alpha_true = 2.01   # Silverberg+2016 GJ 1243
E0 = 1e30           # erg
Emax = 1e34         # erg (upper limit of observed FFD)
N = 6000            # >6000 flares in Silverberg+2016 catalog

energies = sample_power_law_flares(N, alpha_true, E0, Emax, rng)
alpha_hat, sigma_alpha = ml_power_law_index(energies, E0)
print(f"True alpha = {alpha_true:.3f}")
print(f"ML estimate: alpha_hat = {alpha_hat:.4f} +/- {sigma_alpha:.4f}")
print(f"Silverberg+2016 reported: alpha = 2.008 +/- 0.002")

In [ ]:
# Plot cumulative FFD
sorted_e = np.sort(energies)[::-1]
rank = np.arange(1, N + 1)
monitoring_days = 330  # Kepler 11-month baseline in days
cumulative_rate = rank / monitoring_days  # flares/day above each E

fig, ax = plt.subplots()
ax.loglog(sorted_e, cumulative_rate, '.', ms=3, alpha=0.5, label='Simulated GJ 1243-like FFD')
E_grid = np.logspace(np.log10(E0), np.log10(Emax), 200)
ax.loglog(E_grid, (N / monitoring_days) * (E_grid / E0) ** (1.0 - alpha_true),
          'r-', lw=2, label=fr'Power law $\alpha = {alpha_true}$')
ax.set_xlabel('Flare energy $E$ [erg]')
ax.set_ylabel('Cumulative rate $Q(>E)$ [flares / day]')
ax.set_title('Stellar flare FFD — GJ 1243 analog (Silverberg+2016)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 2: White-light flare continuum (9000 K BB + Balmer jump) / 백색광 플레어 연속광

The optical $\lambda \geq 4000$ Å flare continuum fits a $T_{\rm BB} \approx 9000$–$14000$ K blackbody (Hawley & Pettersen 1991; Kowalski et al. 2013). Below the Balmer edge at 3646 Å, a single BB underpredicts the observed flux — optically thin hydrogen bound-free recombination adds a positive jump. We model this as:
$$
F_\lambda^{\rm flare} = \pi B_\lambda(T_{\rm BB}) \cdot \Omega_{\rm flare} + \epsilon_{\rm BJ}(\lambda)
$$
where $\Omega_{\rm flare} = A_{\rm flare}/d^2$ is the flare solid angle, and $\epsilon_{\rm BJ}$ is a step-function approximation to the Balmer continuum.

광학 $\lambda \geq 4000$ Å 연속광은 9000–14000 K 흑체로 잘 맞춰지며, 3646 Å 아래에서는 광학적으로 얇은 수소 재결합 연속광이 양의 점프(Balmer jump)를 더한다.

In [ ]:
def planck_lambda(wavelength_cm: np.ndarray, T: float) -> np.ndarray:
    """Spectral radiance B_lambda(T) in erg s^-1 cm^-2 cm^-1 sr^-1.

    Args:
        wavelength_cm: Wavelength grid in cm.
        T: Blackbody temperature in K.

    Returns:
        Spectral radiance at each wavelength.
    """
    x = H_PLANCK * C_LIGHT / (wavelength_cm * K_BOLTZ * T)
    return (2.0 * H_PLANCK * C_LIGHT ** 2 / wavelength_cm ** 5) / (np.exp(x) - 1.0)


def flare_continuum(wavelength_ang: np.ndarray, T_bb: float, balmer_ratio: float) -> np.ndarray:
    """Simple flare continuum: Planck function + Balmer jump step.

    Args:
        wavelength_ang: Wavelength grid in angstrom.
        T_bb: Blackbody temperature in K.
        balmer_ratio: Observed jump ratio f(3615)/f(4170). 1.0 = pure BB.

    Returns:
        Relative flux per angstrom (arbitrary normalization).
    """
    lam_cm = wavelength_ang * 1e-8
    bb = planck_lambda(lam_cm, T_bb)
    # Add a Balmer continuum enhancement for wavelengths < 3646 A
    # Modeled as an exponential rise with decay scale 1000 A beyond the edge
    balmer_edge = 3646.0
    enhancement = np.where(wavelength_ang < balmer_edge,
                           (balmer_ratio - 1.0) * np.exp(-(balmer_edge - wavelength_ang) / 1000.0),
                           0.0)
    bb_at_4170 = planck_lambda(4170e-8, T_bb)
    return bb + enhancement * bb_at_4170


wavelengths = np.linspace(2500, 9000, 800)
flux_9000K = flare_continuum(wavelengths, T_bb=9000.0, balmer_ratio=2.5)
flux_12000K = flare_continuum(wavelengths, T_bb=12000.0, balmer_ratio=2.0)
flux_bb_only = flare_continuum(wavelengths, T_bb=9000.0, balmer_ratio=1.0)

fig, ax = plt.subplots()
ax.plot(wavelengths, flux_9000K / flux_9000K.max(), 'b-', lw=2, label='9000 K + Balmer jump (GF type)')
ax.plot(wavelengths, flux_12000K / flux_12000K.max(), 'r-', lw=2, label='12000 K + smaller jump (IF type)')
ax.plot(wavelengths, flux_bb_only / flux_bb_only.max(), 'k--', lw=1.5, label='Pure 9000 K BB (no jump)')
ax.axvline(3646, color='gray', ls=':', lw=1, label='Balmer edge 3646 A')
ax.set_xlabel(r'Wavelength [$\mathrm{\AA}$]')
ax.set_ylabel('Normalized flux density')
ax.set_title('Stellar white-light flare continuum: BB + Balmer jump')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: Neupert effect / Neupert 효과

The Neupert effect states that the cumulative nonthermal hard X-ray (HXR) emission tracks the thermal soft X-ray (SXR) emission:
$$
L_{\rm SXR}(t) \propto \int_{-\infty}^{t} L_{\rm HXR}(t')\,dt'
$$
We model an HXR light curve as a fast Gaussian pulse and verify that its time integral matches the SXR rise shape.

Neupert 효과는 누적 HXR 복사가 SXR 복사를 추적한다는 경험 법칙이다. HXR을 빠른 Gaussian으로 모델하고 시간 적분이 SXR 상승과 일치함을 확인한다.

In [ ]:
def neupert_lightcurves(t_s: np.ndarray, t_peak: float, sigma: float,
                       tau_cool: float) -> tuple[np.ndarray, np.ndarray]:
    """Generate paired HXR (impulsive) and SXR (integrated + cooling) light curves.

    Args:
        t_s: Time grid in seconds.
        t_peak: HXR peak time in seconds.
        sigma: HXR Gaussian width in seconds.
        tau_cool: SXR exponential cooling timescale in seconds.

    Returns:
        Tuple (HXR flux, SXR flux), each in arbitrary units.
    """
    hxr = np.exp(-0.5 * ((t_s - t_peak) / sigma) ** 2)
    # SXR = running integral of HXR convolved with exponential decay
    dt = t_s[1] - t_s[0]
    sxr = np.zeros_like(t_s)
    for i in range(1, len(t_s)):
        sxr[i] = sxr[i - 1] * np.exp(-dt / tau_cool) + hxr[i] * dt
    return hxr, sxr


t = np.linspace(0, 600, 2000)
hxr, sxr = neupert_lightcurves(t, t_peak=120.0, sigma=25.0, tau_cool=180.0)

# Time integral of HXR
hxr_integral = integrate.cumulative_trapezoid(hxr, t, initial=0.0)

fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(10, 8))
ax1.plot(t, hxr / hxr.max(), 'r-', lw=2, label='HXR (nonthermal, impulsive)')
ax1.plot(t, sxr / sxr.max(), 'b-', lw=2, label='SXR (thermal)')
ax1.plot(t, hxr_integral / hxr_integral.max(), 'g--', lw=1.5,
         label=r'$\int$ HXR dt (normalized)')
ax1.set_ylabel('Normalized flux')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_title('Neupert effect: SXR tracks the time integral of HXR')

ax2.plot(t, sxr / sxr.max() - hxr_integral / hxr_integral.max(), 'k-', lw=1.5)
ax2.axhline(0, color='gray', ls=':')
ax2.set_xlabel('Time [s]')
ax2.set_ylabel('SXR - integral(HXR)')
ax2.set_title('Residual after cooling-correction')
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Peak HXR at t = {t[np.argmax(hxr)]:.1f} s")
print(f"Peak SXR at t = {t[np.argmax(sxr)]:.1f} s")
print(f"HXR-SXR peak delay (should be >0) = {t[np.argmax(sxr)] - t[np.argmax(hxr)]:.1f} s")

## Part 4: M dwarf vs G dwarf flare rates / M dwarf 대 G dwarf 플레어율

Kepler statistics show M dwarfs (e.g., GJ 1243) flare $>100\times$ more frequently per unit total luminosity than Sun-like G dwarfs. Superflares with $E \geq 10^{33}$ erg occur on Sun-like stars at approximately $\sim 1$/century (Maehara+2012), while on a mid-M dwarf like GJ 1243 comparable energies occur $>10$ times per day.

Kepler 통계에 따르면 M dwarf(GJ 1243)은 태양형 G dwarf보다 단위 광도당 플레어율이 100배 이상 높다. 태양형 별의 $E \geq 10^{33}$ erg 슈퍼플레어는 세기당 1회 수준.

In [ ]:
def cumulative_ffd(E: np.ndarray, alpha: float, E0: float, rate_at_E0: float) -> np.ndarray:
    """Downward-cumulative flare rate above E, per unit time.

    Args:
        E: Energy grid in erg.
        alpha: Differential power-law index.
        E0: Reference energy in erg.
        rate_at_E0: Rate above E0 in flares per day.

    Returns:
        Cumulative rate in flares/day above each energy.
    """
    return rate_at_E0 * (E / E0) ** (1.0 - alpha)


E_grid = np.logspace(29, 37, 300)

# GJ 1243 (M4): alpha = 2.01, observed rate at 1e31 erg ~ 1 flare/day
rate_m4 = cumulative_ffd(E_grid, alpha=2.01, E0=1e31, rate_at_E0=1.0)
# Proxima Centauri (M5.5): alpha ~ 1.7 (Walker 1981)
rate_prox = cumulative_ffd(E_grid, alpha=1.7, E0=1e29, rate_at_E0=0.2)
# Solar-type G dwarf (Sun-like): alpha ~ 1.8 (Shibayama+2013), rate at 1e33 ~ 1/century
rate_g = cumulative_ffd(E_grid, alpha=1.8, E0=1e33, rate_at_E0=1.0 / (100 * 365))

fig, ax = plt.subplots()
ax.loglog(E_grid, rate_prox, 'm-', lw=2, label='Proxima Cen (M5.5, Walker 1981)')
ax.loglog(E_grid, rate_m4, 'r-', lw=2, label='GJ 1243 (M4, Silverberg+2016)')
ax.loglog(E_grid, rate_g, 'b-', lw=2, label='Solar-type G dwarf (Shibayama+2013)')
ax.axvline(1e33, color='gray', ls=':', label='Superflare threshold 1e33 erg')
ax.axhline(1.0 / (100 * 365), color='k', ls='--', alpha=0.5,
           label='1 per century')
ax.set_xlabel('Flare energy $E$ [erg]')
ax.set_ylabel('Cumulative rate $Q(>E)$ [flares / day]')
ax.set_title('M dwarf vs G dwarf flare rates')
ax.set_ylim(1e-8, 1e2)
ax.legend(loc='lower left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Key numbers
e_query = 1e33
print(f"At E = 1e33 erg:")
print(f"  GJ 1243 rate = {cumulative_ffd(e_query, 2.01, 1e31, 1.0):.2e} flares/day")
print(f"  Sun-like G dwarf rate = {cumulative_ffd(e_query, 1.8, 1e33, 1.0/(100*365)):.2e} flares/day")

## Part 5: Habitable-zone UV fluence from Proxima b / 거주가능 영역 UV fluence

Proxima b orbits at $a_p = 0.0485$ au, far closer than Earth's 1 au. Even though Proxima Centauri's quiescent luminosity is only $\sim 0.0017\,L_\odot$, its frequent flares deliver high UV fluence. We compute the flare UV fluence at the planet for a Carrington-scale event ($E_{\rm bol} = 5\times 10^{32}$ erg) vs a Proxima Cen superflare ($10^{33}$ erg).

Proxima b는 0.0485 au에서 공전하여 지구보다 훨씬 가깝다. Proxima Cen의 정지 광도는 $0.0017\,L_\odot$에 불과하지만, 빈번한 플레어가 높은 UV fluence를 행성에 전달한다.

In [ ]:
def flare_fluence_at_planet(flare_energy_erg: float, planet_distance_au: float,
                            uv_fraction: float = 0.3) -> float:
    """Compute UV fluence at a planet from a flare of given bolometric energy.

    Args:
        flare_energy_erg: Bolometric flare energy in erg.
        planet_distance_au: Planet semi-major axis in au.
        uv_fraction: Fraction of flare energy emitted as UV (1200-3200 A).

    Returns:
        UV fluence at the planet in erg/cm^2.
    """
    d_cm = planet_distance_au * AU_CM
    return uv_fraction * flare_energy_erg / (4.0 * np.pi * d_cm ** 2)


# Carrington event (solar) at Earth
carrington_E = 5e32
fluence_earth_carrington = flare_fluence_at_planet(carrington_E, 1.0)

# 2019 Proxima Cen superflare at Proxima b
prox_superflare_E = 1e33
proxb_a = 0.0485
fluence_proxb_superflare = flare_fluence_at_planet(prox_superflare_E, proxb_a)

# Average Prox Cen flare at Proxima b
prox_avg_E = 1e30
fluence_proxb_avg = flare_fluence_at_planet(prox_avg_E, proxb_a)

print("UV fluence comparison:")
print(f"  Carrington event at Earth: {fluence_earth_carrington:.2e} erg/cm^2")
print(f"  Prox Cen 2019 superflare at Proxima b: {fluence_proxb_superflare:.2e} erg/cm^2")
print(f"  Average Prox Cen flare at Proxima b: {fluence_proxb_avg:.2e} erg/cm^2")
print(f"  Ratio Prox b superflare / Earth Carrington: "
      f"{fluence_proxb_superflare / fluence_earth_carrington:.1f}x")

# Visualize
distances = np.logspace(-2, 1, 200)
fluence_grid = np.array([flare_fluence_at_planet(prox_superflare_E, d) for d in distances])
fluence_grid_carr = np.array([flare_fluence_at_planet(carrington_E, d) for d in distances])

fig, ax = plt.subplots()
ax.loglog(distances, fluence_grid, 'm-', lw=2, label='Prox Cen 2019 superflare (1e33 erg)')
ax.loglog(distances, fluence_grid_carr, 'b-', lw=2, label='Solar Carrington (5e32 erg)')
ax.axvline(proxb_a, color='m', ls=':', label=f'Proxima b (a = {proxb_a} au)')
ax.axvline(1.0, color='b', ls=':', label='Earth (a = 1 au)')
ax.set_xlabel('Distance from host star [au]')
ax.set_ylabel('UV fluence [erg / cm^2]')
ax.set_title('Single-flare UV fluence vs orbital distance')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 6: Exoplanet atmospheric photolysis from a superflare / 슈퍼플레어로 인한 대기 광분해

A simplified photochemistry estimate: an ozone column $N_{\rm O_3} \approx 10^{19}$ cm$^{-2}$ (Earth-like) is destroyed when the UV fluence exceeds the ionization/photolysis cross section inverse $\sim 10^{-18}$ cm$^{2}$ × photon energy. We compute the fraction of an Earth-like ozone column photolysed by a single Proxima Cen superflare incident on Proxima b.

간략 광화학 추정: 지구형 오존 기둥($N_{\rm O_3} \approx 10^{19}$ cm$^{-2}$)은 UV fluence가 Proxima Cen 슈퍼플레어 수준에 도달하면 큰 부분이 광분해된다.

In [ ]:
def ozone_depletion_fraction(uv_fluence_erg_cm2: float,
                             ozone_column: float = 1e19,
                             mean_photon_energy_eV: float = 6.2,
                             sigma_o3: float = 1.1e-17) -> float:
    """Estimate ozone column depletion from a UV fluence.

    Args:
        uv_fluence_erg_cm2: UV fluence in erg/cm^2.
        ozone_column: Initial O3 column density in cm^-2.
        mean_photon_energy_eV: Representative Hartley-band photon energy.
        sigma_o3: O3 photolysis cross section in cm^2 (Hartley band peak).

    Returns:
        Fractional depletion of the ozone column (0-1).
    """
    photon_energy_erg = mean_photon_energy_eV * 1.602e-12
    photons_per_cm2 = uv_fluence_erg_cm2 / photon_energy_erg
    tau_photolysis = photons_per_cm2 * sigma_o3
    return 1.0 - np.exp(-tau_photolysis / max(ozone_column * 1e-19, 1.0))


# Apply to the UV fluences computed above
events = {
    'Solar Carrington at Earth': fluence_earth_carrington,
    'Prox Cen avg flare at Proxima b': fluence_proxb_avg,
    'Prox Cen 2019 superflare at Proxima b': fluence_proxb_superflare,
}

print('Ozone column depletion per flare event (simple estimate):')
for name, fluence in events.items():
    frac = ozone_depletion_fraction(fluence)
    print(f'  {name}: fluence = {fluence:.2e} erg/cm^2 -> depletion = {frac*100:.2f}%')

# Cumulative effect: assume 1 superflare per decade on Proxima Cen (Howard+2018)
n_superflares_per_Gyr = 1e8
cumulative_depletion = 1.0 - (1.0 - ozone_depletion_fraction(fluence_proxb_superflare))**100
print(f'\nAfter 100 superflares (~1 kyr): cumulative depletion ~ {cumulative_depletion*100:.2f}%')
print('This illustrates why habitability of M-dwarf planets requires atmospheric regeneration.')
print('(M-dwarf 행성의 거주가능성은 대기 재생 과정 없이는 유지되지 않음을 보여준다.)')

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| FFD power law | $\alpha \approx 2$ for GJ 1243 (dM4) | LSST flare catalog, TESS short-cadence |
| 9000 K BB continuum | Peak-phase blackbody + Balmer jump | RADYN/FCHROMA RHD simulations |
| Neupert effect | HXR integral tracks SXR | Solar Orbiter STIX + EUI comparisons |
| M vs G flare rate | $>100\times$ more frequent on M dwarfs | MUSCLES Treasury UV spectra |
| Proxima b UV fluence | $400\times$ Earth-Carrington | Exoplanet atmospheric escape models |
| Ozone photolysis | Single superflare depletes large fraction | 3D photochemical GCM studies |

이 노트북은 Kowalski (2024) 리뷰의 핵심 정량 개념 — FFD 멱법칙, 백색광 흑체+Balmer jump, Neupert 효과, 행성 UV fluence, 오존 광분해 — 를 단순 재현하여 관측·모델 수치가 어떻게 M dwarf 플레어의 다파장 특성과 외계행성 거주가능성 논의로 이어지는지 보인다.

This notebook reproduces, in simplified form, the quantitative concepts at the heart of the Kowalski (2024) review: the FFD power law, the white-light BB+Balmer-jump continuum, the Neupert effect, stellar-planet UV fluences, and ozone photolysis — linking observational numbers to habitability implications.